# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

# !git clone https://github.com/IgnacioOQ/e_network_inequality
!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

/content/e_network_inequality/e_network_inequality


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/'
print("Current Directory:", dumping_path)

Mounted at /content/drive
Current Directory: /content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/


# Testing Peptic Ulcer

In [ ]:
def generate_parameters_here(_,G,method='randomization'):
    process_seed = int.from_bytes(os.urandom(4), byteorder='little')
    rd.seed(process_seed)
    # Randomly sample parameters for this group
    uncertainty = rd.uniform(.000001, .001)
    n_experiments = rd.randint(1000, 10000)
    # now we pick a random number
    # Capped at 1/3 to prevent "Sample larger than population" errors in equalize
    proportion_edges = rd.rand() * (1/3)
    # Do randomization
    num_edges = G.number_of_edges()
    if method == 'randomization':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = randomize_network(G, n_edges=num_edges_to_randomize)
    if method == 'equalize':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = equalize(G, num_edges_to_randomize)
    if method == 'densify':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='original',keep_density_fixed=False)
    if method == 'densify_fixed':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='uniform',keep_density_fixed=True)
    if method =='cluster':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = cluster_network(G,num_edges_to_add)
    if method =='decluster':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = decluster_network(G,num_edges_to_randomize)

    result = generate_parameters_aggregate(modified_network, uncertainty=uncertainty, n_experiments=n_experiments,
                                           p_rewiring=proportion_edges)
    result['uncertainty'] = uncertainty
    result['n_experiments'] = n_experiments
    result['proportion_edges'] = proportion_edges
    result['parameter_random_seed']= process_seed
    return result

In [ ]:
with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
  G_pud = pickle.load(f)

n_agents = G_pud.number_of_nodes()
print(n_agents)

# Create a mapping from node names to indexes
mapping = {node: index for index, node in enumerate(G_pud.nodes())}

# Relabel the nodes in the graph
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)
G_default = G_pud_indexed

# n_simulations = 100
num_cores = cpu_count()  # Get the number of available CPU cores
print(num_cores)

In [ ]:
n_simulations = 10000
methods = ['randomization','equalize','densify','densify_fixed']
# methods = ['randomization']

def run_method(method):
    print(f'Running stuff for method: {method}')

    # ─── Generate parameters ───
    print('Generating parameters...')
    generate_params = partial(generate_parameters_here, G=G_default, method=method)

    with Pool(num_cores) as pool:
        param_dict = list(tqdm(
            pool.imap_unordered(generate_params, range(n_simulations)),
            total=n_simulations
        ))

    # ─── Run simulations ───
    run_simulation_wrapper = partial(run_vectorized_simulation_with_params,
                                     tolerance=5e-3,
                                     tolerance_stopping=False,
                                     tstep_stopping=True,
                                     number_of_steps=10000)
    with Pool(num_cores) as pool:
        simulation_results = list(tqdm(
            pool.imap_unordered(run_simulation_wrapper, param_dict),
            total=len(param_dict),
            desc="Running simulations"
        ))

    # ─── Save results ───
    results_path = dumping_path + f"pud_results_corrected_{method}.csv"
    new_results_df = pd.DataFrame(simulation_results)

    if os.path.exists(results_path):
        existing_results_df = pd.read_csv(results_path)
        combined_results_df = pd.concat([existing_results_df, new_results_df], ignore_index=True)
    else:
        combined_results_df = new_results_df

    combined_results_df.to_csv(results_path, index=False)
    print(len(combined_results_df), "\n")

# Run each method in isolation
for method in methods:
    run_method(method)

## Basic Plotting Peptic Ulcer

In [ ]:
for method in methods:
  print(f'Plots for method: {method}')
  results_path = dumping_path + "pud_results_corrected_"+method+".csv"
  combined_results_df = pd.read_csv(results_path)
  print(len(combined_results_df))
  scatter_plot(combined_results_df)
  scatter_plot(combined_results_df, target_variable="convergence_step")
  print('\n')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# scatter_plot(combined_results_df)

In [ ]:
# scatter_plot(combined_results_df, target_variable="convergence_step")

## Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))

✅ Disconnected from runtime at: 2025-10-02 11:18:40 EDT


<IPython.core.display.Javascript object>